In [1]:
import os
import json
import xml.etree.ElementTree as ET
import glob
from collections import defaultdict

DATASET_ROOT = "Datasets/AMI_Corpus/ami_public_manual_1.6.2" # root folder
OUTPUT_FILE = "phase1_ami_clean.jsonl" # json output

In [2]:
def parse_transcript_from_xmls(word_files):
    """
    Parses a list of .xml files (one per speaker) and merges them.
    """
    all_words = [] # to add all the spoken word from all the speakers
    
    for xml_file in word_files:
        try:
            # Extracting speaker ID
            filename = os.path.basename(xml_file)
            parts = filename.split('.')
            if len(parts) >= 3:
                speaker = parts[1]
            else:
                speaker = "Unknown"

            # loads XML file converts into tree structure and do extraction
            tree = ET.parse(xml_file)
            root = tree.getroot()

            # extracts start time, end time and text
            for w in root.findall('w'):
                start = w.get('starttime')
                end = w.get('endtime')
                text = w.text
                
                # Only keep words with valid text and timing
                if text and start and end:
                    all_words.append({
                        "start": float(start),
                        "end": float(end),
                        "speaker": speaker,
                        "text": text.strip()
                    })
        except Exception:
            continue

    # Sort by time to create the dialogue flow
    all_words.sort(key=lambda x: x['start'])
    
    # Merge into script format
    transcript = []
    current_speaker = None
    current_line = []
    
    for item in all_words:

        # checks the speaker and if speaker changes saves it and changes the speaker
        if item['speaker'] != current_speaker:
            if current_speaker is not None:
                transcript.append(f"Speaker {current_speaker}: {' '.join(current_line)}")
            current_speaker = item['speaker']
            current_line = [item['text']]
        else:
            current_line.append(item['text'])
            
    if current_speaker and current_line:
        transcript.append(f"Speaker {current_speaker}: {' '.join(current_line)}")
        
    return "\n".join(transcript)

In [3]:
def parse_summary_xml(summary_file):
    """Parses the abstractive summary XML if it exists."""
    try:
        tree = ET.parse(summary_file)
        root = tree.getroot()
        sentences = [elem.text for elem in root.iter() if elem.tag == 'sentence' and elem.text]
        return " ".join(sentences)
    except Exception:
        return None

In [4]:
print(f"Scanning {DATASET_ROOT} for AMI files...")

# adds meeting IDs to their files
meetings = defaultdict(lambda: {'summary': None, 'words': []})

# to find all the xml files
all_xmls = glob.glob(os.path.join(DATASET_ROOT, "**", "*.xml"), recursive=True)

print(f"Found {len(all_xmls)} XML files.")

Scanning Datasets/AMI_Corpus/ami_public_manual_1.6.2 for AMI files...
Found 5080 XML files.


In [5]:
print("Sample files found (to verify patterns):")
for f in all_xmls[:5]:
    print(f" - {os.path.basename(f)}")
# ----------------------------------------------

for f_path in all_xmls:
    fname = os.path.basename(f_path)
    
    # Check for Transcript (Pattern: ES2002a.A.words.xml)
    if 'words.xml' in fname:
        meeting_id = fname.split('.')[0]
        meetings[meeting_id]['words'].append(f_path)
    
    # Check for Summary (Pattern: abstract.xml)
    elif 'abstract.xml' in fname:
        meeting_id = fname.split('.')[0]
        meetings[meeting_id]['summary'] = f_path

print(f"Identified {len(meetings)} unique meetings.")

Sample files found (to verify patterns):
 - AMI-metadata.xml
 - resource.xml
 - ES2002a.abssumm.xml
 - ES2002b.abssumm.xml
 - ES2002c.abssumm.xml
Identified 171 unique meetings.


In [6]:
processed_count = 0
generated_placeholder_count = 0

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
    for meeting_id, data in meetings.items():
        
        # WE ONLY REQUIRE TRANSCRIPTS NOW
        if data['words']:
            transcript_text = parse_transcript_from_xmls(data['words'])
            
            # Handle summaries
            if data['summary']:
                summary_text = parse_summary_xml(data['summary'])
            else:
                summary_text = "TO_BE_GENERATED"
                generated_placeholder_count += 1
            
            if transcript_text:
                entry = {
                    "source": "AMI",
                    "id": meeting_id,
                    "input": transcript_text,
                    "output": summary_text
                }
                f_out.write(json.dumps(entry) + "\n")
                processed_count += 1

print("-" * 30)
print(f"Done! Saved {processed_count} meetings to {OUTPUT_FILE}")
print(f" - {processed_count - generated_placeholder_count} have original summaries.")
print(f" - {generated_placeholder_count} marked as 'TO_BE_GENERATED' (Teacher LLM will fix this).")

------------------------------
Done! Saved 171 meetings to phase1_ami_clean.jsonl
 - 0 have original summaries.
 - 171 marked as 'TO_BE_GENERATED' (Teacher LLM will fix this).
